# Dataset Overview

Data sources, filtering pipeline, and feature definitions for the midfielder transfer performance models.

## 1. Data Sources

| Source | What it provides |
|--------|------------------|
| **Twelve Football (Wyscout)** | Player performance metrics, 20 Twelve Quality Scores, team tactical stats |
| **Transfermarkt** | Transfer records, fees, market values |

Two parquet files:
- `within_league_transfers_v5.parquet` — One row per player-transfer with pre/post qualities and team stats
- `team_qualities.parquet` — One row per team-competition-season with 7 tactical style dimensions

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
print(f"Full dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

Full dataset: 18,065 rows, 120 columns


## 2. Filtering Pipeline

From the master dataset (262K transfers) to the modelling-ready v5:

| Step | Filter | Effect |
|------|--------|--------|
| 1 | Same league | Within-league transfers only |
| 2 | Different team | No same-team re-entries |
| 3 | Same position pre/post | Consistent role comparison |
| 4 | 900+ minutes both sides | Sufficient playing time |
| 5 | No goalkeepers | 5 outfield positions only |

In [2]:
print("Position breakdown:")
print(df["from_position"].value_counts().to_string())
print(f"\nSeasons: {sorted(df['from_season'].unique())}")

Position breakdown:
from_position
Midfielder          5230
Central Defender    4766
Full Back           3812
Striker             2422
Winger              1835

Seasons: [2018, 2019, 2020, 2021, 2022, 2023, 2024]


## 3. Midfielder Subset

In [3]:
mf = df[df["from_position"] == "Midfielder"].copy()
print(f"Midfielders: {mf.shape[0]:,} rows")

Midfielders: 5,230 rows


## 4. Features

### 4.1 Player Quality Scores (17 of 20 used)

Twelve Football computes 20 composite quality scores per player. Each row has `from_Q` (pre-transfer) and `to_Q` (post-transfer).

3 excluded for midfielders (>50% null): Chance prevention, Territorial dominance, Poaching.

In [4]:
QUALITIES = [
    "Active defence", "Aerial threat", "Box threat", "Composure",
    "Defensive heading", "Dribbling", "Effectiveness", "Finishing",
    "Hold-up play", "Intelligent defence", "Involvement",
    "Passing quality", "Pressing", "Progression",
    "Providing teammates", "Run quality", "Winning duels",
]

from_pq = [f"from_{q}" for q in QUALITIES]
to_pq = [f"to_{q}" for q in QUALITIES]

excluded = ["Chance prevention", "Territorial dominance", "Poaching"]
for q in excluded:
    print(f"{q}: {mf[f'from_{q}'].isna().mean()*100:.0f}% null")

Chance prevention: 100% null
Territorial dominance: 100% null
Poaching: 75% null


### 4.2 Team Tactical Qualities (7 dimensions)

| Dimension | What it captures |
|-----------|------------------|
| Defence | Defensive solidity |
| Defensive Transition | Recovering possession |
| Attacking Transition | Counter-attacking |
| Attack | Sustained possession-based attacking |
| Penetration | Breaking into the box |
| Chance Creation | Creating shooting opportunities |
| Outcome | Converting chances into goals/points |

- `from_q_proj_*` — Origin team, projected to destination season's league distribution
- `to_q_*` — Destination team, in own season's league distribution

Both z-scored against the same reference, making them directly comparable.

In [5]:
TEAM_Q = ["DEFENCE", "DEFENSIVE_TRANSITION", "ATTACKING_TRANSITION",
          "ATTACK", "PENETRATION", "CHANCE_CREATION", "OUTCOME"]

from_tq = [f"from_q_proj_{q}" for q in TEAM_Q]
to_tq = [f"to_q_{q}" for q in TEAM_Q]
delta_tq = [f"delta_team_{q}" for q in TEAM_Q]

### 4.3 Derived Features (Deltas)

- **Delta team quality:** `delta_team_Q_k = to_team_k - from_team_proj_k` (change in tactical environment)

## 5. Clean Dataset & Train/Test Split

In [6]:
for q in TEAM_Q:
    mf[f"delta_team_{q}"] = mf[f"to_q_{q}"] - mf[f"from_q_proj_{q}"]

all_cols = from_pq + to_pq + from_tq + to_tq + delta_tq + ["from_season"]
mf_clean = mf[all_cols].dropna()

train = mf_clean[mf_clean["from_season"] <= 2023]
test = mf_clean[mf_clean["from_season"] == 2024]

print(f"Clean midfielders: {len(mf_clean):,}")
print(f"Train (2018-2023): {len(train):,}")
print(f"Test (2024):       {len(test):,}")

Clean midfielders: 4,848
Train (2018-2023): 4,383
Test (2024):       465


## 6. What Goes Into Each Model

All models predict `to_Q_i`. Models 4 and 5 internally predict the delta but are evaluated on the reconstructed `to_Q_i = from_Q_i + predicted_delta`.

| # | Model | Features | Predicts |
|---|-------|----------|----------|
| 1 | **Naive Baseline** | `from_Q_i` (1) | `to_Q_i` directly |
| 2 | **Player Profile** | `from_Q_1..17` (17) | `to_Q_i` directly |
| 3 | **Player + Team Context** | `from_Q_1..17` + `from_team_1..7` + `to_team_1..7` (31) | `to_Q_i` directly |
| 4 | **Tactical Shift** | `delta_team_1..7` (7) | `delta_Q_i` → `to_Q_i` |
| 5 | **Player + Tactical Shift** | `from_Q_1..17` + `delta_team_1..7` (24) | `delta_Q_i` → `to_Q_i` |